# PreProcessing du csv supervise

## Initialisation

In [10]:
import pandas as pd
import cv2
import numpy as np
import json
from sklearn.metrics.pairwise import cosine_similarity
import os

In [11]:
df = pd.read_csv('../../kidneyData.csv')

##### Pour etre conforme a notre version nous allons retirer `/content/data` du CSV

In [12]:
df['path'] = df['path'].str.replace('/content/data/', '')
print(df['path'])

0        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
1        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
2        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
3        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
4        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
                               ...                        
12441    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
12442    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
12443    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
12444    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
12445    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
Name: path, Length: 12446, dtype: str


## Suppression des column inutile

In [13]:
df = df.iloc[:, 1:]
df = df.drop('path', axis=1)
df = df.drop('Class', axis=1)
print(df)

            image_id   diag  target
0      Tumor- (1044)  Tumor       3
1        Tumor- (83)  Tumor       3
2       Tumor- (580)  Tumor       3
3      Tumor- (1701)  Tumor       3
4      Tumor- (1220)  Tumor       3
...              ...    ...     ...
12441   Cyst- (2522)   Cyst       0
12442   Cyst- (2627)   Cyst       0
12443    Cyst- (972)   Cyst       0
12444   Cyst- (2323)   Cyst       0
12445   Cyst- (2145)   Cyst       0

[12446 rows x 3 columns]


## Ajout des array dans la column matrice 

Ancien test

In [ ]:
# df = pd.read_csv("../../Supervise.csv")

# taille_standard = (128, 128)

# dossiers = {
#     "Tumor": "../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Tumor/",
#     "Normal": "../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Normal/",
#     "Stone": "../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Stone/",
#     "Cyst": "../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Cyst/"
# }

# toutes_les_matrices = []

# # on passe dans chaque dossier
# for dossier in lst_dossier:
#     # boucle pour parcourir tout le dossier
#     for i, nom_fichier in enumerate(os.listdir(dossier)):
#         # on combimne le chemin avec le nom du fichier pour avoir le bon link
#         chemin_complet = os.path.join(dossier, nom_fichier)
        
#         # lecture de l'img
#         img = cv2.imread(chemin_complet)
        
#         # on passe en gris les images pour regler les problemes de rgb
#         gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
#         #redimension
#         img_resized = cv2.resize(gray, taille_standard)

#         # on ajoute dans notre iste d'array l'array qu'on vient de recuperer
#         toutes_les_matrices.append(img_resized.flatten())

        

# # on stocke dans matrice l'array de toute les images
# # df["matrice"] = toutes_les_matrices
# df = pd.DataFrame({"matrice": toutes_les_matrices})
# df['matrice'] = df['matrice']
# print(df['matrice'])



Pour faire cela, nous allons passer dans les 4 dossiers et dans chaque image des dossiers.
On va appliquer une redimension de 128 par 128 tout en garddant les proportions.
Nous allons egalement gerer le probleme des couleurs et basculer toute les image en gris.
Une fois cela fait on ajoute l'array numpy de la photo directement dans la ligne correspondante dans l'ordre

In [10]:
taille_standard = (128, 128)

dossiers = {
    "Tumor": "../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Tumor/",
    "Normal": "../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Normal/",
    "Stone": "../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Stone/",
    "Cyst": "../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Cyst/"
}

toutes_les_matrices = []

for _, row in df.iterrows():

    dossier = dossiers[row["diag"]]

    nom_fichier = row["image_id"] + ".jpg"

    chemin_complet = os.path.join(dossier, nom_fichier)

    img = cv2.imread(chemin_complet)

    if img is None:
        print("Erreur :", chemin_complet)
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    img_resized = cv2.resize(gray, taille_standard)

    toutes_les_matrices.append(img_resized.flatten())

X = np.array(toutes_les_matrices)

np.save("../../Matrice.npy", X)

print("X shape :", X.shape)
print("CSV :", len(df))

FileNotFoundError: [Errno 2] No such file or directory: '../../Supervise.csv'

##### Justification de l'utilisation du module CV2 dans le fichier `JustificationCV2` a la racine du dossier actuel

##### Apres les echecs avec les reultat du premier model on va donc creer des groupes d'images similaire et les ajouter au csv.

In [15]:
# on charge notre fichier de matrice
matrices = np.load('../../Matrice.npy', allow_pickle=True)

# on applati les matrice pour les comparer + facilement
flat = matrices.reshape(matrices.shape[0], -1)

# mesure la direction entre deux valeurs (images) et donne un chiffre de similitude
sim = cosine_similarity(flat)
# on trouve les similaire mais on prend l'image elle meme
np.fill_diagonal(sim, 0)

prctRessemblance = 0.995
paris_P_Identique = np.argwhere(sim > prctRessemblance)


In [17]:
def find(parent, i):
        """
        Permet de remonter dans tout les parents jusqu'a se trouver elle meme.
        Si on a [0, 0, 1, 4]
        on cherche find(parent, 1), on obtiens 0 donc on fait find(parent, 0) et on obteiens 0 (la meme image que le i) donc on a termine on est retombe desus, alors 1 et 0 sont les meme

        Prend en argument parent et i:
            - i est l'image qu'on cherche
            - parent est une liste qui contient globalement qui pointe vers qui, imaginons image A pointe vers l'image B alors dans parent on aura B, et ca se cumule avec les autres enfants

        return i
        """
        # tant qu'one est pas sur le meme
        while parent[i] != i:
            i = parent[i]
        return i

def union(parent, i, j):
    """
    Prend en parametre parent i et j:
        - parent est la liste qui contient qui pointe vers qui
        - i et j sont les images qu'on veut fusionner
    """
    lst_i = find(parent, i)
    list_j = find(parent, j)
    # on verif qu'ils sont pas deja dans le meme groupe
    if lst_i != list_j:
        # on fait pointer i vers j qui devient son parent
        parent[list_j] = lst_i


# on prend la premiere column
n = sim.shape[0]

# creer le tableau parent de base, au debut chaque image pointe vers elle meme car elel est dientique a elle meme
parent = np.arange(n)


# on va faire des paris d'images similaire
pairs = np.argwhere(sim > prctRessemblance)
# (voir en bas pour l'explication)

# on passe dans chacune des pairs pour trouver les unions avec les autres paris
for i, j in pairs:
    union(parent, i, j)


# on va stoccker tt les resultats 
resultat = []
for i in range(n):
    resultat.append(find(parent, i))


In [20]:
matrices = np.load('../../Matrice.npy', allow_pickle=True)

df['groupe'] = resultat

In [21]:
print(df)

            image_id   diag  target  groupe
0      Tumor- (1044)  Tumor       3       0
1        Tumor- (83)  Tumor       3      52
2       Tumor- (580)  Tumor       3     624
3      Tumor- (1701)  Tumor       3     927
4      Tumor- (1220)  Tumor       3       4
...              ...    ...     ...     ...
12441   Cyst- (2522)   Cyst       0   12441
12442   Cyst- (2627)   Cyst       0   12442
12443    Cyst- (972)   Cyst       0   12443
12444   Cyst- (2323)   Cyst       0    8770
12445   Cyst- (2145)   Cyst       0   12445

[12446 rows x 4 columns]


## Export de notre dataset en CSV

In [22]:
df.to_csv("../../Supervise.csv")